# Part 1 — Identifying CYP substrates

**The task.** Decide which of the 1,223 screened compounds are substrates of CYP3A4 and CYP2J2,
*with statistical rationale*.

**The short version of what follows.** The released summary table cannot support that decision on
its own — it reports a ratio of means with no uncertainty attached. Working from the well-level
data instead turns up three properties of this dataset that change the answer:

1. complete depletion is recorded as `area = 0`, and those are the *strongest* substrates;
2. control and treatment wells are not position-matched on the plate;
3. unreactive compounds do not sit at 100% remaining — there is a systematic ~15% offset.

Each is verified below before anything is built on it. I've kept one wrong turn in
(section 4.2) because the mistake is instructive and it changed a conclusion.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
from scipy import stats as st

from octant_cyp import io, qc, normalize, substrates, stats as ocstats

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

## 1. Why not just use the provided `pct_remaining`?

The release ships a compound×enzyme summary. Let's look at what it actually contains.

In [2]:
summary = io.load_summary()
print(summary.shape)
summary.head(3)

(2446, 9)


,ocnt_batch,enzyme,control,treatment,log10_control,log10_treatment,log10fc,log2fc,pct_remaining
0,OCNT-0017486-AA-002,CYP2J2,18836.875,20992.000,4.275032,4.322075,0.047043,0.156272,111.440990
1,OCNT-0017486-AA-002,CYP3A4,26667.000,27727.625,4.425990,4.442928,0.016938,0.056266,103.977294
2,OCNT-0021863-AA-002,CYP2J2,18422.750,4664.375,4.265378,3.668887,-0.596491,-1.981502,25.318560


Two things stand out immediately: **there are no structures** (so any chemistry in Part 2 has to
come from elsewhere), and **there is no uncertainty** — no standard error, no interval, no p-value.

Before trusting these columns, let me confirm exactly how they were computed rather than assuming.

In [3]:
wells = io.load_wells()
means = (wells.groupby(["ocnt_batch", "enzyme", "condition"])["area"]
              .mean().unstack().reset_index())
means.columns = ["ocnt_batch", "enzyme", "ctrl_mean", "trt_mean"]
chk = summary.merge(means, on=["ocnt_batch", "enzyme"])

print("control  == mean of control wells   :", np.allclose(chk.control, chk.ctrl_mean))
print("treatment== mean of treatment wells  :", np.allclose(chk.treatment, chk.trt_mean))
print("pct_remaining == 100*treat/control   :",
      np.allclose(summary.pct_remaining, 100 * summary.treatment / summary.control))
print("log2fc / log10fc is a constant ratio :",
      round(float((summary.log2fc / summary.log10fc).std()), 12),
      "(=> same quantity, different base; ratio %.4f)" % (summary.log2fc / summary.log10fc).mean())

control  == mean of control wells   : True
treatment== mean of treatment wells  : True
pct_remaining == 100*treat/control   : True
log2fc / log10fc is a constant ratio : 0.0 (=> same quantity, different base; ratio 3.3219)


So `pct_remaining` is a **ratio of two means**, and `log2fc`/`log10fc` are that same number
rescaled. Nothing here carries information about *how variable* the four replicates were.

That matters because a fixed ">50% depleted" rule then treats these two compounds identically:

- one whose four replicates agree to within 3%, sitting at 55% remaining;
- one whose replicates scatter across 20–90%, averaging 55% remaining.

The first is confidently *not* a hit at that threshold; the second is uninformative. To separate
them we need the replicates, which means working from the well-level file.

## 2. The well-level data

One row per well: 4 replicates per compound × enzyme × condition, plus the SMILES the summary lacks.

In [4]:
print(wells.shape)
print("\ncompounds:", wells.ocnt_batch.nunique(), "| unique SMILES:", wells.standardized_smiles.nunique())
print("\nreplicates per (compound, enzyme, condition):")
print(wells.groupby(["ocnt_batch", "enzyme", "condition"]).size().value_counts().to_string())
wells.head(3)

(19344, 12)

compounds: 1223 | unique SMILES: 1223

replicates per (compound, enzyme, condition):
4    4671
3     218
2       3


,ocnt_batch,enzyme,condition,plate,well,time_start,time_end,mz_query,mz_observed,mass_error_ppm,area,standardized_smiles
0,OCNT-0454895-AA-001,CYP3A4,control,Plate01,A01,0.0075,0.03,462.238733,462.238313,-0.908799,19596.0,CCOC(=O)N1CCC(NC2=C(N(CCC3=CC=CC=C3)CC3=CC=CC=...
1,OCNT-0457791-AA-001,CYP3A4,control,Plate01,A01,0.0075,0.03,380.171716,380.171941,0.590579,13677.5,CC(=O)NC1=CC=CC(C2=NC(C3=CC=CC(NC(=O)NC(C)C)=C...
2,OCNT-0493944-AA-001,CYP3A4,control,Plate01,A01,0.0075,0.03,330.196474,330.195929,-1.651426,96551.0,CN(C)C1=CC=C(C=NN(CC2=CC=CC=C2)C2=CC=CC=C2)C=C1


1,223 compounds map 1:1 to 1,223 unique structures. Replication is mostly n=4 but **not uniformly** —
218 groups have 3 and a few have 2, so nothing downstream may assume balanced design.

## 3. Discovery 1 — the zeros are data, not gaps

848 wells report `area = 0`. The obvious reading is "missing measurement". Let's test that.

In [5]:
print(qc.censoring_summary(wells).to_string(index=False))
print("\nmass_error_ppm is NaN in exactly the rows where area == 0:",
      bool(((wells.area == 0) == wells.mass_error_ppm.isna()).all()))

enzyme condition  n_zero    n  pct_zero
CYP2J2   control       0 4815  0.000000
CYP2J2 treatment      70 4873  1.436487
CYP3A4   control       0 4829  0.000000
CYP3A4 treatment     778 4827 16.117671

mass_error_ppm is NaN in exactly the rows where area == 0: True


Decisive. **Every zero is on the treatment side; no control well is ever zero.** And the mass error
is missing precisely when the area is zero — i.e. no peak was found at the expected m/z.

That is not a missing measurement. It is the parent compound being metabolised below the detection
limit: a **left-censored observation**, and the single most informative outcome in the assay.

How often does it happen?

In [6]:
pattern = qc.censoring_pattern(wells)
print(pattern.pattern.value_counts().to_string())
print("\nby enzyme:")
print(pd.crosstab(pattern.enzyme, pattern.pattern).to_string())

pattern
none       2191
full        148
partial     107

by enzyme:
pattern  full  none  partial
enzyme                      
CYP2J2      8  1199       16
CYP3A4    140   992       91


148 compound×enzyme pairs have **every** treatment replicate censored, and 107 more are partially
censored. Here is why that matters concretely — what a routine log-scale pipeline does to them:

In [7]:
naive = summary.assign(log_fc=np.log10(summary.treatment / summary.control))
dropped = naive[~np.isfinite(naive.log_fc)]
print(f"compounds silently lost to log(0): {len(dropped)}")
print("their pct_remaining values:", sorted(dropped.pct_remaining.unique()))
print("\nSo the compounds discarded are exactly the ones at 0% remaining —")
print("the most completely metabolised compounds in the screen.")

compounds silently lost to log(0): 148
their pct_remaining values: [np.float64(0.0)]

So the compounds discarded are exactly the ones at 0% remaining —
the most completely metabolised compounds in the screen.


/Users/kevinerazocastillo/anaconda3/envs/cyp-reactivity/lib/python3.11/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


This is the first real design decision: **model the censoring** rather than drop or fudge it.
`octant_cyp.stats.fit_censored_contrast` fits a Tobit (left-censored normal) MLE on `log10(area)`.

The detection limit is taken from the data — the smallest peak area the pipeline ever reported —
rather than assumed:

In [8]:
lod = io.estimate_lod(wells)
det = wells.loc[wells.area > 0, "area"]
print(f"LOD (smallest reported area) = {lod}")
print("control-well areas never approach it:  min =", wells.loc[wells.condition=='control','area'].min())
print("\nlow tail of *detected* areas:", np.percentile(det, [0.1, 1, 5]).round(1))

LOD (smallest reported area) = 21.0
control-well areas never approach it:  min = 2824.0

low tail of *detected* areas: [  26.2   75.  2003.4]


Controls bottom out at 2,824 against an LOD of 21, so censoring is a treatment-side phenomenon
only — consistent with it being metabolism rather than an instrument problem.

### Does the Tobit fit actually help?

Rather than assert it, simulate: generate data with a known effect where replicates straddle the
LOD, then compare the censored MLE against the obvious shortcut of dropping non-detects.

In [9]:
rng = np.random.default_rng(11)
lod_s, mu_c, true_delta, sigma = 21.0, 4.5, -3.1, 0.30
naive_err, tobit_err = [], []
for _ in range(600):
    ctrl = 10 ** rng.normal(mu_c, sigma, 4)
    yt = rng.normal(mu_c + true_delta, sigma, 4)
    trt = np.where(10 ** yt < lod_s, 0.0, 10 ** yt)
    if (trt == 0).sum() in (0, 4):
        continue
    fit = ocstats.fit_censored_contrast(ctrl, trt, lod=lod_s)
    tobit_err.append(fit.delta - true_delta)
    obs = trt[trt > 0]
    naive_err.append(np.log10(obs).mean() - np.log10(ctrl).mean() - true_delta)

print(f"simulated partially-censored compounds: {len(tobit_err)}")
print(f"  drop non-detects : bias = {np.mean(naive_err):+.3f} log10  <- under-states depletion")
print(f"  censored MLE     : bias = {np.mean(tobit_err):+.3f} log10")

simulated partially-censored compounds: 516
  drop non-detects : bias = +0.181 log10  <- under-states depletion
  censored MLE     : bias = -0.023 log10


Dropping non-detects biases the estimate *towards less depletion* — it throws away the lowest
readings, so the surviving mean is too high. The censored fit removes most of that bias.
(This is pinned as a unit test in `tests/test_stats.py`.)

## 4. Discovery 2 — the plate layout is not position-matched

The assay blog flags edge effects as something modellers should check. Checking it turned up
something more consequential than edge effects.

In [10]:
print("plate geometry (rows, cols):", qc.plate_geometry(wells))
print()
print(qc.condition_layout(wells).to_string(index=False))

plate geometry (rows, cols): (32, 48)

condition  row_min  row_max  n_rows_used  col_min  col_max  n_wells
  control        1        4            4        1       48     9644
treatment        5       32           28        1       48     9700


**Controls occupy only rows A–D; treatments occupy rows E–AF.** The two arms are spatially
segregated, so the control/treatment contrast is *not* position-matched, and any top-to-bottom
plate gradient maps directly onto every fold-change.

One thing does work in our favour — each compound's control and treatment wells sit on the *same*
plate, so plate-to-plate effects cancel inside the ratio:

In [11]:
spans = wells.groupby(["ocnt_batch", "enzyme"]).plate.nunique()
print("compound-enzyme pairs spanning more than one plate:", int((spans > 1).sum()), "of", len(spans))
print("=> control and treatment are co-plated; plate effects cancel within a compound's ratio.")
print()
cv = qc.control_cv(wells)
print(f"assay noise floor — control CV: median {cv.cv.median():.3f}, "
      f"IQR {cv.cv.quantile(.25):.3f}-{cv.cv.quantile(.75):.3f}")

compound-enzyme pairs spanning more than one plate: 0 of 2446
=> control and treatment are co-plated; plate effects cancel within a compound's ratio.

assay noise floor — control CV: median 0.080, IQR 0.054-0.110


## 4.2 Discovery 3 — unreactive compounds don't sit at 100%

This is where the analysis nearly went wrong, so I've left the wrong turn in.

Start with a fact that has to be true: incubating a compound with an enzyme cannot *create* parent
compound. Anything above 100% remaining is noise or artefact. Now look at CYP2J2:

In [12]:
j = summary[summary.enzyme == "CYP2J2"]
print(f"CYP2J2 median pct_remaining: {j.pct_remaining.median():.1f}%")
print(f"fraction of compounds appearing ENRICHED by incubation: {100*(j.pct_remaining > 100).mean():.1f}%")

CYP2J2 median pct_remaining: 109.9%
fraction of compounds appearing ENRICHED by incubation: 62.5%


62.5% of CYP2J2 compounds appear *enriched*. That is not biologically possible, so there is a
systematic offset between the control and treatment arms — plausibly a consequence of the
spatial segregation found above, or of matrix differences between the two well populations.

To correct it we need to locate where the **inactive population** actually sits. My first attempt
used a KDE with scipy's default relative bandwidth:

In [13]:
grid = np.linspace(-8, 3, 2001)
print("Locating the inactive mode with a shared *relative* bandwidth (bw_method=0.2):\n")
for enz in ["CYP2J2", "CYP3A4"]:
    x = summary.loc[summary.enzyme == enz, "log2fc"].replace([np.inf, -np.inf], np.nan).dropna().values
    dens = st.gaussian_kde(x, bw_method=0.2)(grid)
    win = (grid > -0.6) & (grid < 1.0)
    mode = grid[win][np.argmax(dens[win])]
    print(f"  {enz}: sd={x.std():.2f}  ->  mode {mode:+.3f} log2 = {100*2**mode:.1f}% remaining")

Locating the inactive mode with a shared *relative* bandwidth (bw_method=0.2):

  CYP2J2: sd=2.01  ->  mode +0.189 log2 = 114.0% remaining
  CYP3A4: sd=5.25  ->  mode -0.471 log2 = 72.2% remaining


Read at face value, that says CYP2J2 has an inactive population at ~115% but **CYP3A4 does not have
one at all** — its apparent mode sits at 72% remaining, which would be absurd to treat as "no change".
I concluded the CYP3A4 offset was unidentifiable and left that arm uncorrected.

That conclusion was an artefact of my own smoothing. `bw_method` is a multiplier on the *sample
standard deviation*, and the two enzymes have very different spreads — CYP3A4's distribution is
dominated by a broad active lobe, so the same relative bandwidth smooths it far harder and erases
the narrow inactive peak. Using a fixed **absolute** bandwidth instead:

In [14]:
print("Same data, matched *absolute* bandwidth (0.08 log2 units):\n")
for enz in ["CYP2J2", "CYP3A4"]:
    x = summary.loc[summary.enzyme == enz, "log2fc"].replace([np.inf, -np.inf], np.nan).dropna().values
    mode = normalize.inactive_mode(x)
    sep = normalize.mode_separation(x)
    print(f"  {enz}: sd={x.std():.2f}  ->  mode {mode:+.3f} log2 = {100*2**mode:.1f}% remaining "
          f"(peak/trough {sep:,.0f})")

Same data, matched *absolute* bandwidth (0.08 log2 units):

  CYP2J2: sd=2.01  ->  mode +0.261 log2 = 119.8% remaining (peak/trough 11,866)
  CYP3A4: sd=5.25  ->  mode +0.230 log2 = 117.3% remaining (peak/trough 24)


Both enzymes show the **same** offset, ~+0.2 log2 (≈115% of control). That is what an artefact of
the well layout should look like — a property of the plate, not of the enzyme — and it is much more
credible than "one enzyme has no inactive compounds".

`tests/test_normalize.py` now contains a regression test asserting that two distributions with
identical offsets but different spreads return the same mode.

The offset is estimated per plate, with a bootstrap to show how precisely:

In [15]:
plate_map = wells[["ocnt_batch", "enzyme", "plate"]].drop_duplicates()
offsets = normalize.estimate_plate_offsets(summary, plate_map)
print(offsets[["enzyme", "plate", "offset_log2", "offset_pct_of_control",
               "mode_separation", "identifiable"]].round(3).to_string(index=False))
print()
for enz, g in summary.groupby("enzyme"):
    x = g.log2fc.replace([np.inf, -np.inf], np.nan).dropna().values
    b = normalize.bootstrap_mode(x, n_boot=300)
    print(f"  {enz}: {b['mode']:+.3f} log2   95% CI [{b['ci_low']:+.3f}, {b['ci_high']:+.3f}]")

enzyme   plate  offset_log2  offset_pct_of_control  mode_separation  identifiable
CYP2J2 Plate05        0.262                119.947     1.363405e+12          True
CYP2J2 Plate06        0.221                116.538     1.362777e+12          True
CYP2J2 Plate07        0.274                120.882     1.519768e+12          True
CYP2J2 Plate08        0.264                120.080     1.363376e+12          True
CYP3A4 Plate01        0.254                119.284     2.715660e+02          True
CYP3A4 Plate02        0.133                109.642     2.455160e+02          True
CYP3A4 Plate03        0.245                118.493     2.268080e+02          True
CYP3A4 Plate04        0.227                117.056     7.748100e+01          True



  CYP2J2: +0.261 log2   95% CI [+0.240, +0.279]


  CYP3A4: +0.230 log2   95% CI [+0.022, +0.289]


Consistent across all eight plates. Note the honest asymmetry: the **CYP2J2 estimate is tight**
(±0.02) while **CYP3A4's is much wider** (±0.13), because CYP3A4 has far fewer inactive compounds to
anchor on. That uncertainty is real and is reported rather than smoothed over — and the uncorrected
analysis is retained as a sensitivity arm at the end.

## 5. The model

Per compound × enzyme, on `log10(area)`:

- **Tobit MLE** for the control-vs-treatment contrast, with treatment non-detects left-censored at the LOD.
- **Empirical-Bayes variance moderation** (Smyth 2004). With n=4 the per-compound variance estimate
  has ~6 degrees of freedom and is nearly worthless; moderation borrows strength across all ~1,200
  compounds assayed on the same plates. Implemented and unit-tested in `src/`, not black-boxed.
- **Per-plate offset** subtracted, from the section above.
- **BH FDR** across compounds within each enzyme.

In [16]:
contrasts = substrates.fit_all_contrasts(wells, lod=lod)
usable = offsets[offsets.identifiable]
offset_map = dict(zip(usable.plate, usable.offset_log2))
called = substrates.call_substrates(contrasts, offsets=offset_map)

for enz, g in called.groupby("enzyme"):
    print(f"{enz}: s2_prior={g.s2_prior.iloc[0]:.5f}  df_prior={g.df_prior.iloc[0]:.2f}  "
          f"(per-compound residual df = {g.df.median():.0f})")
    mde = substrates.minimum_detectable_effect(g.s2_prior.iloc[0], g.v.median(), g.df_total.median())
    print(f"   minimum detectable effect @95% power: {100*mde:.1f}% remaining "
          f"({100*(1-mde):.1f}% depletion)")

CYP2J2: s2_prior=0.00152  df_prior=2.98  (per-compound residual df = 6)
   minimum detectable effect @95% power: 77.1% remaining (22.9% depletion)
CYP3A4: s2_prior=0.00266  df_prior=1.52  (per-compound residual df = 6)
   minimum detectable effect @95% power: 70.2% remaining (29.8% depletion)


The moderation adds roughly 1.5–3 degrees of freedom per test. Note the **minimum detectable
effect**: about 30% depletion for CYP3A4 at 95% power. That number drives the decision rule.

## 6. The decision rule, and why it has three classes

My first attempt was the conventional screening rule — significant *and* big enough — plus a
two-sided TOST to certify non-substrates. The TOST returned **zero** confident CYP3A4
non-substrates, which is worth understanding rather than working around.

With a typical standard error of ~0.036 log10 and ~7 df, the 95% interval half-width is ~22% on the
ratio scale. Certifying that a compound lies within ±20% of no change is therefore asking for a
precision the design does not have. **n=4 can find substrates; it cannot certify non-substrates at
a ±20% margin.** That is a property of the experiment, not a flaw in the test.

So instead of forcing a binary call, compounds are partitioned by where the confidence interval
sits relative to a single effect boundary (20% depletion):

| CI position | call |
|---|---|
| entirely below the boundary | **substrate** |
| entirely above | **non-substrate** |
| straddling it | **inconclusive** |

Two one-sided tests, each FDR-controlled, mutually exclusive by construction. The third class is
the point: "not a hit" otherwise conflates *confidently unreactive* with *never adequately
measured*, and Part 3 needs the first as negatives while excluding the second.

In [17]:
for enz, g in called.groupby("enzyme"):
    vc = g.call.value_counts()
    print(f"{enz}: " + ", ".join(f"{k}={v} ({100*v/len(g):.1f}%)" for k, v in vc.items()))

CYP2J2: non-substrate=601 (49.1%), inconclusive=330 (27.0%), substrate=292 (23.9%)
CYP3A4: substrate=884 (72.3%), inconclusive=214 (17.5%), non-substrate=125 (10.2%)


Compounds whose treatment arm is *entirely* censored get a **bounded** estimate: the treatment mean
is pinned at the LOD — the largest value consistent with seeing nothing — so the reported depletion
is a floor, not a point estimate.

In [18]:
a4 = called[called.enzyme == "CYP3A4"].sort_values("delta_adj")
cols = ["ocnt_batch", "call", "pct_remaining_est", "pct_remaining_ci_low",
        "pct_remaining_ci_high", "q_substrate", "censoring"]
print("most-depleted CYP3A4 compounds (effect is a lower bound where censoring == 'full'):")
print(a4[cols].head(5).round(4).to_string(index=False))
print("\nconfident non-substrates:")
print(a4[a4.call == "non-substrate"].nsmallest(4, "se")[
    ["ocnt_batch", "pct_remaining_est", "pct_remaining_ci_low",
     "pct_remaining_ci_high", "q_nonsubstrate"]].round(3).to_string(index=False))

most-depleted CYP3A4 compounds (effect is a lower bound where censoring == 'full'):
         ocnt_batch      call  pct_remaining_est  pct_remaining_ci_low  pct_remaining_ci_high  q_substrate censoring
OCNT-0461740-AA-001 substrate             0.0050                0.0043                 0.0058          0.0      full
OCNT-0461674-AA-001 substrate             0.0050                0.0039                 0.0064          0.0      full
OCNT-0461684-AA-001 substrate             0.0058                0.0012                 0.0286          0.0   partial
OCNT-0461770-AA-001 substrate             0.0068                0.0059                 0.0078          0.0      full
OCNT-0456142-AA-001 substrate             0.0070                0.0058                 0.0084          0.0      full

confident non-substrates:
         ocnt_batch  pct_remaining_est  pct_remaining_ci_low  pct_remaining_ci_high  q_nonsubstrate
OCNT-0476778-AA-002            104.310                94.602                115.013    

## 7. Validation

Four independent checks. The first is the one that matters most: **does this pipeline reproduce the
published result when asked the published question?**

In [19]:
anchor = (summary.assign(hit=summary.pct_remaining < 50)
                 .groupby("enzyme").hit.mean() * 100)
print("Blog anchor — uncorrected, >50% depletion rule:")
print(anchor.round(1).to_string())
print("\nOpenADMET report ~61% (CYP3A4) and ~13% (CYP2J2). Match.")

Blog anchor — uncorrected, >50% depletion rule:
enzyme
CYP2J2    13.6
CYP3A4    61.4

OpenADMET report ~61% (CYP3A4) and ~13% (CYP2J2). Match.


In [20]:
null = substrates.permutation_null(wells, lod=lod, n_rep=2, seed=0)
ks = st.kstest(null.p_value.dropna(), "uniform")
print("Permutation null — split each compound's CONTROL wells into pseudo-arms.")
print("No enzyme is present, so any 'depletion' found is noise.\n")
print(f"  tests            : {len(null)}")
print(f"  fraction p < 0.05: {(null.p_value < 0.05).mean():.4f}   (expect ~0.05)")
print(f"  discoveries q<.05: {(null.q_value < 0.05).sum()}")
print(f"  KS vs uniform    : p = {ks.pvalue:.3f}")

Permutation null — split each compound's CONTROL wells into pseudo-arms.
No enzyme is present, so any 'depletion' found is noise.

  tests            : 4616
  fraction p < 0.05: 0.0409   (expect ~0.05)
  discoveries q<.05: 0
  KS vs uniform    : p = 0.015


Slightly **conservative** (4.1% rather than 5%) and zero false discoveries across 4,616 tests.
The KS test does reject uniformity at p=0.015 — with this many points it is sensitive to small
departures — but the deviation is in the safe direction. Reported rather than hidden.

Finally, the sensitivity analysis: how much of the difference from the published counts is due to
the **effect threshold**, and how much to the **offset correction**?

In [21]:
sens = substrates.threshold_sensitivity(contrasts, offset_map)
print(sens.to_string(index=False))

enzyme    normalisation  effect_threshold_pct_remaining    n  substrate  non_substrate  inconclusive  pct_substrate
CYP2J2      uncorrected                            50.0 1223        132           1020            71           10.8
CYP3A4      uncorrected                            50.0 1223        682            392           149           55.8
CYP2J2      uncorrected                            80.0 1223        221            838           164           18.1
CYP3A4      uncorrected                            80.0 1223        839            217           167           68.6
CYP2J2 offset-corrected                            50.0 1223        152            975            96           12.4
CYP3A4 offset-corrected                            50.0 1223        728            346           149           59.5
CYP2J2 offset-corrected                            80.0 1223        292            601           330           23.9
CYP3A4 offset-corrected                            80.0 1223        884 

Reading the CYP3A4 rows: at the blog's own 50% threshold the corrected substrate rate is 59.5%
versus 55.8% uncorrected — the correction moves us *closer* to their published 61.4%, which is a
mild independent sign that correcting is the right call. Our headline numbers differ from theirs
mainly because we require the whole confidence interval to clear the boundary, which is stricter
than comparing a point estimate.

## 8. Deliverable

`results/substrates_cyp3a4.csv` and `results/substrates_cyp2j2.csv` — per compound: effect size,
95% CI, p and q for both one-sided tests, censoring status, and the three-way call.

**Summary of the argument.** The provided `pct_remaining` cannot distinguish a precise measurement
from a noisy one. Working from wells lets us (a) treat complete depletion as the informative
observation it is rather than dropping the best compounds, (b) remove a systematic ~15% offset that
would otherwise make unreactive compounds look mildly active, and (c) attach an interval to every
compound so the call can be graded rather than forced.